# argentina.economia — Pruebas internas

**Este notebook es interno**, no está pensado para publicarse.
Sirve como recorrido paso a paso de las funcionalidades del módulo `argentina.economia`
en su estado actual (v0.0.3, 366 series en catálogo).

Requiere conexión a internet (la API es `apis.datos.gob.ar`).

## 1. Setup e imports

In [ ]:
import argentina as arg
import pandas as pd

from argentina.economia import (
    SERIES,
    obtener_serie,
    serie,
    ipc_nacional,
    emae,
    tipo_cambio_minorista,
)

print(f"argentina v{arg.__version__}")
print(f"pandas v{pd.__version__}")

## 2. Estructura del catálogo

Cuántas series hay y cómo están organizadas.

In [ ]:
print(f"Total de aliases en el catálogo: {len(SERIES)}")
print(f"IDs únicos: {len({e['id'] for e in SERIES.values()})}")

In [ ]:
# Distribución por tema
from collections import Counter
temas = Counter(e['tema'] for e in SERIES.values())
pd.Series(temas).sort_values(ascending=False)

In [ ]:
# Distribución por frecuencia
frecs = Counter(e['frecuencia'] for e in SERIES.values())
pd.Series(frecs).sort_values(ascending=False)

In [ ]:
# Distribución por fuente (top 5)
fuentes = Counter(e['fuente'] for e in SERIES.values())
pd.Series(fuentes).sort_values(ascending=False).head()

## 3. Inspeccionar una entrada del catálogo

In [ ]:
SERIES['ipc_nacional']

In [ ]:
SERIES['emae']

## 4. Wrappers de las series macro principales

Las tres funciones ya exportadas: `ipc_nacional`, `emae`, `tipo_cambio_minorista`.

In [ ]:
ipc = ipc_nacional(start_date='2020-01-01')
ipc.head()

In [ ]:
ipc.tail()

In [ ]:
# EMAE
emae_df = emae(start_date='2020-01-01')
emae_df.head()

In [ ]:
# Tipo de cambio minorista (diario)
tc = tipo_cambio_minorista(start_date='2024-01-01')
print(f"{len(tc)} observaciones diarias")
tc.tail()

## 5. Bajar por alias del catálogo expandido

Cualquiera de los 366 aliases sirve.

In [ ]:
# Buscar aliases que contengan "emae"
[a for a in SERIES if 'emae' in a][:15]

In [ ]:
# EMAE desestacionalizada
emae_desest = serie('emae_desestacionalizada', start_date='2020-01-01')
emae_desest.head()

In [ ]:
# Nivel general de servicios públicos
serv_pub = serie('nivel_general', start_date='2015-01-01')
serv_pub.tail()

## 6. Bajar por ID directo (sin pasar por alias)

Útil si tenés un ID que no está en el catálogo local.

In [ ]:
# Misma serie del IPC pero pasando el ID
df = obtener_serie('148.3_INIVELNAL_DICI_M_26', start_date='2023-01-01')
df.head()

## 7. Filtrar el catálogo por palabra clave

Búsqueda ad-hoc dentro del catálogo local (sin red).

In [ ]:
def buscar_local(palabra: str) -> pd.DataFrame:
    palabra = palabra.lower()
    filas = []
    for alias, e in SERIES.items():
        haystack = f"{alias} {e['descripcion']} {e.get('dataset','')}".lower()
        if palabra in haystack:
            filas.append({
                'alias': alias,
                'id': e['id'],
                'frecuencia': e.get('frecuencia', ''),
                'tema': e.get('tema', ''),
                'descripcion': e['descripcion'][:80],
            })
    return pd.DataFrame(filas)

buscar_local('salario')

In [ ]:
buscar_local('construccion').head(10)

In [ ]:
buscar_local('petroleo')

## 8. Combinar varias series en un solo DataFrame

In [ ]:
def juntar(aliases, start_date=None):
    dfs = []
    for a in aliases:
        df = serie(a, start_date=start_date).rename(columns={'valor': a})
        dfs.append(df.set_index('fecha'))
    return pd.concat(dfs, axis=1).reset_index()

panel = juntar(['ipc_nacional', 'emae'], start_date='2020-01-01')
panel.head()

## 9. Análisis exploratorio: variación interanual del IPC

In [ ]:
ipc_full = ipc_nacional(start_date='2018-01-01')
ipc_full = ipc_full.set_index('fecha')
ipc_full['var_interanual_%'] = ipc_full['valor'].pct_change(12) * 100
ipc_full.tail(12)

## 10. Casos borde: respuesta vacía

Si pedimos un rango sin datos, la función devuelve un DataFrame vacío con las columnas correctas (no rompe).

In [ ]:
# Rango futuro absurdo
vacio = obtener_serie('148.3_INIVELNAL_DICI_M_26', start_date='2050-01-01', end_date='2050-12-31')
print(f"vacío? {vacio.empty}")
print(f"columnas: {list(vacio.columns)}")
vacio

## 11. Tests automáticos

Los tests unitarios mockean `requests.get` (no usan red) y verifican:

1. Que `serie(alias)` resuelve al ID correcto.
2. Que `obtener_serie` convierte el JSON a DataFrame con columnas `fecha` y `valor`.
3. Que respuesta vacía → DataFrame vacío con esquema correcto.
4. Que el catálogo tiene ≥ 360 entradas con campos completos.
5. Que los IDs del catálogo son únicos.

Para correrlos:

```bash
cd /Users/tobiasyatche/argentina
pytest -v
```

## Notas sueltas / TODOs

- El endpoint `/search/` de datos.gob.ar no indexa todas las series descargables. Los wrappers macro (`ipc_nacional`, `emae`, `tipo_cambio_minorista`) usan IDs verificados a mano.
- Faltarían sumar IDs verificados de BCRA (reservas, BADLAR, tasa política monetaria), más variantes de tipo de cambio (mayorista, CCL), e IPC desagregado por región.
- `buscar_local` no está expuesta como API pública todavía — vive solo en este notebook. Si la usás seguido, conviene moverla a `economia/__init__.py`.
- Sin caché ni descarga masiva por ahora (a propósito, para mantener simple).